# 预归一化
$$
y = \frac{x}{\sqrt{\frac{1}{n}\sum_{i=1}^{n}x_i^2 + \epsilon}} \cdot \gamma
$$
接口：x: 输入张量，形状为 (batch_size, seq_length, dim)  
     𝛾: 可学习的缩放参数，形状为 (hidden_size,)  
     𝜖: 防止除零的小常数，默认为 1e-12

In [ ]:
import torch
import torch.nn as nn
class RMSNorm(nn.Module):
    def __init__(self,dim: int, eps:float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = torch.ones(dim)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm_x * self.weight

# 旋转位置编码（相对位置编码）
$(r_1 e^{i\theta_1}) \cdot (r_2 e^{i\theta_2}) = r_1 r_2 e^{i(\theta_1 + \theta_2)}$


In [ ]:
import torch
def precompute_freqs_cis(dim:int,end:int,theta:float=10000):
    freqs = 1.0/(theta**(torch.arange(0,dim,2)/2))
    pos = torch.arange(0,end)
    freqs = torch.outer(pos,freqs) ## (end,dim/2)算角度
    freqs_cis = torch.polar(torch.ones_like(freqs),freqs) ## (end,dim/2),复数表示
    return freqs_cis

def reshape_for_broadcast(x:torch.Tensor,freqs_cis:torch.Tensor):
    ndim = x.dim#(batch,seq_len,num_heads,head_dim/2)
    shape = (d if i==1 or i==ndim-1 else 1 for i,d in enumerate(x.shape))
    freqs_cis = freqs_cis.view(*shape) ##(1,seq_len,1,head_dim/2)
    return freqs_cis

def apply_rotary_emb(x_q:torch.Tensor,x_k:torch.Tensor,freqs_cis:torch.Tensor):
    xq_ = torch.view_as_complex(x_q.float().reshape(*x_q.shape[:-1],-1,2)) #(batch,seq_len,num_heads,head_dim/2),复数形式
    xk_ = torch.view_as_complex(x_k.float().reshape(*x_k.shape[:-1],-1,2))
    freqs_q = reshape_for_broadcast(x_q,freqs_cis)
    freqs_k = reshape_for_broadcast(x_k,freqs_cis)
    xq_out = torch.view_as_real(xq_*freqs_q).flatten(-2) #(batch,seq_len,num_heads,head_dim)
    xk_out = torch.view_as_real(xk_*freqs_k).flatten(-2)
    return xq_out.type_as(x_q),xk_out.type_as(x_k)



# 分组查询注意力
<strong>多组query共享一个k,v</strong>

In [ ]:
import torch.nn as nn
def repeat_kv(x:torch.Tensor,n_rep:int):
    batch_size,seq_len,num_heads,head_dim = x.shape
    if n_rep == 1:
        return x
    return(x[:,:,:,None,:].expand(batch_size,seq_len,num_heads,n_rep,head_dim).reshape(batch_size,seq_len,num_heads*n_rep,head_dim))

class GroupedQueryAttention(nn.Module):
    def __init__(self,dim:int,n_heads:int,n_kv_heads:int):
        super().__init__()
        self.n_heads = n_heads
        self.dim = dim
        self.head_dim = dim // n_heads
        self.n_kv_heads = n_kv_heads
        self.rep = n_heads // n_kv_heads
        self.to_q = nn.Linear(dim,n_heads * self.head_dim,bias=False)
        self.to_k = nn.Linear(dim,n_kv_heads * self.head_dim,bias=False)
        self.to_v = nn.Linear(dim,n_kv_heads * self.head_dim,bias=False)
        self.fc = nn.Linear(n_heads * self.head_dim,dim)
    def forward(self,x:torch.Tensor,x_kv:torch.Tensor,mask:torch.Tensor=None):
        q = self.to_q(x).view(x.shape[0],x.shape[1],self.n_heads,self.head_dim)#(batch,seq_len,n_heads,head_dim)
        k = self.to_k(x_kv).view(x_kv.shape[0],x_kv.shape[1],self.n_kv_heads,self.head_dim)#(batch,seq_len,n_kv_heads,head_dim)
        v = self.to_v(x_kv).view(x_kv.shape[0],x_kv.shape[1],self.n_kv_heads,self.head_dim)
        k = repeat_kv(k,self.rep)
        v = repeat_kv(v,self.rep)
        scores = torch.matmul(q.transpose(1,2),k.transpose(1,2).transpose(-2,-1)) / (self.head_dim ** 0.5) #(batch,n_heads,seq_len,seq_len)
        if mask is not None:
            scores = scores.masked_fill(mask==0,float('-inf'))
        attention = torch.softmax(scores,dim=-1)
        out = torch.matmul(attention,v.transpose(1,2)) #(batch,n_heads,seq_len,head_dim)
        out = out.transpose(1,2).contiguous().view(x.shape[0],x.shape[1],self.n_heads * self.head_dim)
        out = self.fc(out)
        return out

# SwiGLU激活函数
- 使用三个线性变换（`W`, `V`, `W2`）而不是两个。
- 第一个变换 `xW` 经过 Swish 激活函数（`swish(x) = x * sigmoid(x)`）。
- 第二个变换 `xV` 作为“门”，与前一步的结果进行逐元素相乘。
- 最后通过第三个变换 `W2` 输出。

公式如下，其中 $\otimes$ 是逐元素乘法：

$$
\text{SwiGLU}(x, W, V, W_2) = (\text{swish}(xW) \otimes xV)W_2
$$

In [ ]:
class FeedForward(nn.Module):
    def __init__(self,dim:int,hidden_dim:int,multi_of:int):
        super().__init__()
        hidden_dim = int(hidden_dim * 2/3)
        hidden_dim = multi_of * ((hidden_dim + multi_of -1)//multi_of)
        self.w1 = nn.Linear(dim,hidden_dim)
        self.w2 = nn.Linear(hidden_dim,dim)
        self.w3 = nn.Linear(dim,hidden_dim)
    def forward(self,x:torch.Tensor):
        return self.w2(torch.nn.functional.silu(self.w1(x)) * self.w3(x))

# 模型组装

In [ ]:
import torch
class TransformerBlock(nn.Module):
    def __init__(self,dim:int,hidden_dim:int,n_heads:int,n_kv_heads:int,multi_of:int):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attention = GroupedQueryAttention(dim,n_heads,n_kv_heads)
        self.norm2 = RMSNorm(dim)
        self.ffn = FeedForward(dim,hidden_dim,multi_of)
        self.norm = RMSNorm(dim)
    def forward(self,x:torch.Tensor,mask:torch.Tensor=None): ## 预归一化
        x = x + self.attention(self.norm1(x),self.norm1(x),mask)
        x = x + self.ffn(self.norm2(x))
        return x

class LlamaModel(nn.Module):
    def __init__(self,vocab_size:int,dim:int,hidden_dim:int,n_heads:int,n_kv_heads:int,n_layers:int,max_seq_len:int,multi_of:int):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size,dim)
        self.layers = nn.ModuleList([TransformerBlock(dim,hidden_dim,n_heads,n_kv_heads,multi_of) for _ in range(n_layers)])
        self.norm = RMSNorm(dim)
        self.output = nn.Linear(dim,vocab_size,bias=False)
        self.register_buffer('freqs_cis',precompute_freqs_cis(dim//n_heads,max_seq_len))
    def forward(self,input:torch.Tensor,start_pos:int=0):
        x = self.token_emb(input)
        freqs_cis = self.freqs_cis[start_pos:start_pos + x.shape[1]]
        mask = None
        seq_len = x.shape[1]
        if seq_len > 1:
            mask = torch.full((seq_len,seq_len),float('-inf'))
            mask = torch.triu(mask,diagonal=1).to(x.device)
            mask = torch.hstack([torch.zeros((seq_len,start_pos),device=x.device),mask])
        for layer in self.layers:
            x = layer(x,start_pos,freqs_cis,mask)
        x = self.norm(x)
        x = self.output(x)
        return x
    


In [23]:
import torch
import torch.nn as nn
class RMSNorm(nn.Module):
    def __init__(self,dim: int, eps:float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm_x * self.weight
    

def precompute_freqs_cis(dim:int,end:int,theta:float=10000):
    freqs = 1.0/(theta**(torch.arange(0,dim,2)/dim))
    pos = torch.arange(0,end)
    freqs = torch.outer(pos,freqs) ## (end,dim/2)算角度
    freqs_cis = torch.polar(torch.ones_like(freqs),freqs) ## (end,dim/2),复数表示
    return freqs_cis##end=

def reshape_for_broadcast(x:torch.Tensor,freqs_cis:torch.Tensor)->torch.Tensor:
    ndim = x.dim() #(batch,seq_len,num_heads,head_dim/2)
    shape = [1] * ndim
    shape[1] = x.shape[1]
    shape[-1] = x.shape[-1]
    freqs_cis = freqs_cis[:x.shape[1]]  # (actual_seq_len, D//2)
    return freqs_cis.view(*shape)


def apply_rotary_emb(x_q:torch.Tensor,x_k:torch.Tensor,freqs_cis:torch.Tensor):
    xq_ = torch.view_as_complex(x_q.float().reshape(*x_q.shape[:-1],-1,2)) #(batch,seq_len,num_heads,head_dim/2),复数形式
    xk_ = torch.view_as_complex(x_k.float().reshape(*x_k.shape[:-1],-1,2))
    freqs_q = reshape_for_broadcast(xq_,freqs_cis)
    freqs_k = reshape_for_broadcast(xk_,freqs_cis)
    xq_out = torch.view_as_real(xq_*freqs_q).flatten(-2) #(batch,seq_len,num_heads,head_dim)
    xk_out = torch.view_as_real(xk_*freqs_k).flatten(-2)
    return xq_out.type_as(x_q),xk_out.type_as(x_k)

import torch.nn as nn
def repeat_kv(x:torch.Tensor,n_rep:int) -> torch.Tensor:
    batch_size,seq_len,num_heads,head_dim = x.shape
    if n_rep == 1:
        return x
    return(x[:,:,:,None,:].expand(batch_size,seq_len,num_heads,n_rep,head_dim).reshape(batch_size,seq_len,num_heads*n_rep,head_dim))

class GroupedQueryAttention(nn.Module):
    def __init__(self,dim:int,n_heads:int,n_kv_heads:int):
        super().__init__()
        self.n_heads = n_heads
        self.dim = dim
        self.head_dim = dim // n_heads
        self.n_kv_heads = n_kv_heads
        self.rep = n_heads // n_kv_heads
        self.to_q = nn.Linear(dim,n_heads * self.head_dim,bias=False)
        self.to_k = nn.Linear(dim,n_kv_heads * self.head_dim,bias=False)
        self.to_v = nn.Linear(dim,n_kv_heads * self.head_dim,bias=False)
        self.fc = nn.Linear(n_heads * self.head_dim,dim)
    def forward(self,x:torch.Tensor,freqs_cis:torch.Tensor,mask):
        q = self.to_q(x).view(x.shape[0],x.shape[1],self.n_heads,self.head_dim)#(batch,seq_len,n_heads,head_dim)
        k = self.to_k(x).view(x.shape[0],x.shape[1],self.n_kv_heads,self.head_dim)#(batch,seq_len,n_kv_heads,head_dim)
        v = self.to_v(x).view(x.shape[0],x.shape[1],self.n_kv_heads,self.head_dim)
        q,k = apply_rotary_emb(q,k,freqs_cis)
        k = repeat_kv(k,self.rep)
        v = repeat_kv(v,self.rep)
        scores = torch.matmul(q.transpose(1,2),k.transpose(1,2).transpose(-2,-1)) / (self.head_dim ** 0.5) #(batch,n_heads,seq_len,seq_len)
        if mask is not None:
            scores = scores+mask
        attention = torch.softmax(scores,dim=-1)
        out = torch.matmul(attention,v.transpose(1,2)) #(batch,n_heads,seq_len,head_dim)
        out = out.transpose(1,2).contiguous().view(x.shape[0],x.shape[1],self.n_heads * self.head_dim)
        out = self.fc(out)
        return out

class FeedForward(nn.Module):
    def __init__(self,dim:int,hidden_dim:int,multi_of:int):
        super().__init__()
        hidden_dim = int(hidden_dim * 2/3)
        hidden_dim = multi_of * ((hidden_dim + multi_of -1)//multi_of)
        self.w1 = nn.Linear(dim,hidden_dim)
        self.w2 = nn.Linear(hidden_dim,dim)
        self.w3 = nn.Linear(dim,hidden_dim)
    def forward(self,x:torch.Tensor):
        return self.w2(torch.nn.functional.silu(self.w1(x)) * self.w3(x))
    
import torch
class TransformerBlock(nn.Module):
    def __init__(self,dim:int,n_heads:int,n_kv_heads:int,multi_of:int,ffn_dim_multiplier:float=None,eps:float=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attention = GroupedQueryAttention(dim,n_heads,n_kv_heads)
        self.norm2 = RMSNorm(dim)
        self.ffn = FeedForward(dim,int(dim*4),multi_of)
        self.norm = RMSNorm(dim)
    def forward(self,x:torch.Tensor,start_pos:int=0,freqs_cis:torch.Tensor=None,mask:torch.Tensor=None): ## 预归一化
        x = x + self.attention(self.norm1(x),freqs_cis,mask)
        x = x + self.ffn(self.norm2(x))
        return x

class LlamaModel(nn.Module):
    def __init__(self,vocab_size:int,dim:int,n_heads:int,n_kv_heads:int,n_layers:int,max_seq_len:int,multi_of:int,ffn_dim_multiplier:float=None,norm_eps:float=1e-6,max_batch_size:int=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size,dim)
        self.layers = nn.ModuleList([TransformerBlock(dim,n_heads,n_kv_heads,multi_of) for _ in range(n_layers)])
        self.norm = RMSNorm(dim)
        self.output = nn.Linear(dim,vocab_size,bias=False)
        self.register_buffer('freqs_cis',precompute_freqs_cis(dim//n_heads,max_seq_len))
    def forward(self,input:torch.Tensor,start_pos:int=0):
        x = self.token_emb(input)
        freqs_cis = self.freqs_cis[start_pos:start_pos + x.shape[1]]
        mask = None
        seq_len = x.shape[1]
        if seq_len > 1:
            mask = torch.full((seq_len,seq_len),float('-inf'))
            mask = torch.triu(mask,diagonal=1).to(x.device)
            mask = torch.hstack([torch.zeros((seq_len,start_pos),device=x.device),mask])
        for layer in self.layers:
            x = layer(x,start_pos,freqs_cis,mask)
        x = self.norm(x)
        x = self.output(x)
        return x
    
def main() -> None:
    # 使用小尺寸参数，便于 CPU/GPU 都能快速跑通
    model = LlamaModel(
        vocab_size=1000,
        dim=256,
        n_layers=2,
        n_heads=8,
        n_kv_heads=2,
        multi_of=64,
        ffn_dim_multiplier=None,
        norm_eps=1e-6,
        max_batch_size=4,
        max_seq_len=64,
    )

    # 构造随机 token 序列并执行前向
    batch_size, seq_len = 2, 16
    tokens = torch.randint(0, 1000, (batch_size, seq_len))
    logits = model(tokens, start_pos=0)

    # 期望: [batch_size, seq_len, vocab_size]
    print("logits shape:", tuple(logits.shape))

if __name__ == "__main__":
    main()

logits shape: (2, 16, 1000)
